# Sharpening metacells

See `INSTALL.md` for installing what this needs. Nothing here installs anything: if the cell below
fails, the environment is not set up, and the notebook says so rather than working around it.

In [1]:
import numpy as np

import dafpy as dp
import metacellspy as mc
import somegraphspy as sg

print("dafpy", dp.__version__)
print("somegraphspy", sg.__version__)
print("metacellspy", mc.__version__)

# Read from Julia at import, so printing it means Python reached Julia rather than merely that the
# Python packages are installed.
print("regularization", mc.GENE_FRACTION_REGULARIZATION_FOR_CELLS)

Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython


[ Info: Will cache ispath data forever
[ Info: Old linux kernel, will pre-populate mmap into RAM disk: Linux version 4.18.0-553.30.1.el8_10.x86_64 (mockbuild@x64-builder01.almalinux.org) (gcc version 8.5.0 20210514 (Red Hat 8.5.0-22) (GCC)) #1 SMP Tue Nov 26 02:30:26 EST 2024


dafpy 0.3.0
somegraphspy 0.2.0
metacellspy 0.1.0
regularization 0.0001


## Importing the cells


In [2]:
# What to take out of the `AnnData`, and under what name. Anything not named here is copied as it is,
# after the importer's own renaming: a `something_cell` or `something_gene` mask arrives as
# `is_something`, and a `something_umis` as `something_UMIs`. Naming a property here overrides that
# for it alone, so the rest of the import is unaffected.
COPY_DATA = {
    # Which metacell each cell belongs to. This is what the pipeline sharpens rather than computes,
    # and it is the one property the importer skips by default, since it usually comes from a
    # separate metacells file. Here the cells are the only place it exists, so we ask for it.
    "metacell_name": ("metacell", None),
    # The type of each cell. **Specify this whenever the data has a type per cell**: the type axis is
    # built from a vector called `type`, and the column holding it is rarely called that. Leave it
    # out and everything still runs, with no types and uncolored graphs.
    "cell_type": ("type", None),
    # This dataset has a column of its own called `type` - the platform each cell was measured on,
    # which is not a cell type at all. Left alone it would collide with the line above.
    "type": ("platform", None),
    #
    # The batch, the plate it was on, and the run it was sequenced in. These become axes of their
    # own further down, so they are given the names those axes will have. Three other columns hold
    # the same batch identifier - `batch_set_id` is identical to it, `plate` is it with 1212 cells
    # saying the literal string `NA`, and `Plate` is it with the 10x cells left blank - so they are
    # dropped rather than imported and then explained.
    "amp_batch_id": ("batch", None),
    "batch_set_id": None,
    "Plate": None,
    "plate": None,
    "Plate..": ("plate", None),
    "seq_batch_id": ("sequencing_run", None),
    #
    # The wet lab's record of each batch, plate and run. These are spelled as they were typed into a
    # spreadsheet, dots, capitals, typos and all, and are about to become properties of those axes
    # where they will be read rather than merely stored.
    "Comment": ("comment", None),
    "Conc...ng.ul.": ("concentration_ng_per_ul", None),
    "Evarage.size..bp.": ("average_size_bp", None),
    "External.Index": ("external_index", None),
    "Internal.Index": ("internal_index", None),
    "QC1": ("qc1", None),
    "QC2": ("qc2", None),
    "delta_CT": ("delta_ct", None),
    "Libprep.Cycles": ("libprep_cycles", None),
    "Owner": ("owner", None),
    "Plate.Date": ("plate_date", None),
    "Production.Date": ("production_date", None),
    "Sort.Date": ("sort_date", None),
    "Last.sequensing.date": ("last_sequencing_date", None),
    "Sequencing.Dates": ("sequencing_dates", None),
    "Experiment": ("experiment", None),
    # Not a genotype: the values are free text describing the sample the batch was made from - the
    # strain, the stage, which embryos, whether it is placenta - and only some of them are strains.
    # It is constant per batch and per plate, and *not* per embryo, which is the giveaway.
    "Genotype": ("description", None),
    #
    # Columns which hold one value, or none at all: `Ref` is `mm10` for every cell that has it,
    # `Empty.Wells` is one list of wells repeated, `X.1` is the string `NA` for all 110,746 cells,
    # `X` is blank for most of them, and `not_na` is true throughout. None of them distinguishes
    # anything, so none of them is worth carrying.
    "Ref": None,
    "Empty.Wells": None,
    "X": None,
    "X.1": None,
    "not_na": None,
}

cells = dp.files_daf("dafs/cells", "w", name="cells")

mc.import_cells_h5ad(cells, cells_h5ad="input/assigned_cells.h5ad", copy_data=COPY_DATA)

print(cells.description())

┌ Debug: Daf: FilesDaf cells path: dafs/cells
└ @ DataAxesFormats.FilesFormat ~/anaconda3/envs/metacells-sharpening/share/julia/packages/DataAxesFormats/vefDL/src/files_format.jl:409
┌ Debug: Metacells.AnnDataFormat.import_cells_h5ad! {
└ @ Metacells.AnnDataFormat ~/anaconda3/envs/metacells-sharpening/share/julia/packages/Metacells/vXiuC/src/anndata_format.jl:168
┌ Debug: - daf: FilesDaf cells
└ @ Metacells.AnnDataFormat ~/anaconda3/envs/metacells-sharpening/share/julia/packages/Metacells/vXiuC/src/anndata_format.jl:168
┌ Debug: - cells_h5ad: "input/assigned_c..." (25)
└ @ Metacells.AnnDataFormat ~/anaconda3/envs/metacells-sharpening/share/julia/packages/Metacells/vXiuC/src/anndata_format.jl:168
┌ Debug: - copy_data: 31 x Str => Union{Nothing, Tuple{AbstractString, Union{Nothing, Bool, Float32, Float64, Int16, Int32, Int64, Int8, UInt16, UInt32, UInt64, UInt8, AbstractString}}} (Dict)
└ @ Metacells.AnnDataFormat ~/anaconda3/envs/metacells-sharpening/share/julia/packages/Metacells/vXiuC

┌ Debug: skip gene vector: full_gene_index
└ @ Metacells.AnnDataFormat ~/anaconda3/envs/metacells-sharpening/share/julia/packages/Metacells/vXiuC/src/anndata_format.jl:573
┌ Debug: copy gene vector: properly_sampled_gene to: is_properly_sampled
└ @ Metacells.AnnDataFormat ~/anaconda3/envs/metacells-sharpening/share/julia/packages/Metacells/vXiuC/src/anndata_format.jl:599
┌ Debug: copy gene vector: selected_gene to: is_selected
└ @ Metacells.AnnDataFormat ~/anaconda3/envs/metacells-sharpening/share/julia/packages/Metacells/vXiuC/src/anndata_format.jl:599
┌ Debug: copy gene vector: bursty_lonely_gene to: is_bursty_lonely
└ @ Metacells.AnnDataFormat ~/anaconda3/envs/metacells-sharpening/share/julia/packages/Metacells/vXiuC/src/anndata_format.jl:599
┌ Debug: copy gene vector: excluded_gene to: is_excluded
└ @ Metacells.AnnDataFormat ~/anaconda3/envs/metacells-sharpening/share/julia/packages/Metacells/vXiuC/src/anndata_format.jl:599
┌ Debug: copy gene vector: lateral_genes_module to: latera

┌ Debug: copy cell vector: ssc_a
└ @ Metacells.AnnDataFormat ~/anaconda3/envs/metacells-sharpening/share/julia/packages/Metacells/vXiuC/src/anndata_format.jl:597
┌ Debug: copy cell vector: pacific_blue_a
└ @ Metacells.AnnDataFormat ~/anaconda3/envs/metacells-sharpening/share/julia/packages/Metacells/vXiuC/src/anndata_format.jl:597
┌ Debug: copy cell vector: time
└ @ Metacells.AnnDataFormat ~/anaconda3/envs/metacells-sharpening/share/julia/packages/Metacells/vXiuC/src/anndata_format.jl:597
┌ Debug: copy cell vector: gfp_a
└ @ Metacells.AnnDataFormat ~/anaconda3/envs/metacells-sharpening/share/julia/packages/Metacells/vXiuC/src/anndata_format.jl:597
┌ Debug: copy cell vector: apc_a
└ @ Metacells.AnnDataFormat ~/anaconda3/envs/metacells-sharpening/share/julia/packages/Metacells/vXiuC/src/anndata_format.jl:597
┌ Debug: copy cell vector: cells_rare_gene_module to: rare_gene_module
└ @ Metacells.AnnDataFormat ~/anaconda3/envs/metacells-sharpening/share/julia/packages/Metacells/vXiuC/src/annd

┌ Debug: copy cell vector: pe_cy5_a
└ @ Metacells.AnnDataFormat ~/anaconda3/envs/metacells-sharpening/share/julia/packages/Metacells/vXiuC/src/anndata_format.jl:597
┌ Debug: copy cell vector: delta_CT to: delta_ct
└ @ Metacells.AnnDataFormat ~/anaconda3/envs/metacells-sharpening/share/julia/packages/Metacells/vXiuC/src/anndata_format.jl:599
┌ Debug: copy cell vector: amp_batch_id to: batch
└ @ Metacells.AnnDataFormat ~/anaconda3/envs/metacells-sharpening/share/julia/packages/Metacells/vXiuC/src/anndata_format.jl:599
┌ Debug: copy cell vector: Internal.Index to: internal_index
└ @ Metacells.AnnDataFormat ~/anaconda3/envs/metacells-sharpening/share/julia/packages/Metacells/vXiuC/src/anndata_format.jl:599
┌ Debug: copy cell vector: Plate.Date to: plate_date
└ @ Metacells.AnnDataFormat ~/anaconda3/envs/metacells-sharpening/share/julia/packages/Metacells/vXiuC/src/anndata_format.jl:599
┌ Debug: copy cell vector: Sort.Date to: sort_date
└ @ Metacells.AnnDataFormat ~/anaconda3/envs/metacells-

┌ Debug: copy cell vector: cell_type to: type
└ @ Metacells.AnnDataFormat ~/anaconda3/envs/metacells-sharpening/share/julia/packages/Metacells/vXiuC/src/anndata_format.jl:599
┌ Debug: copy cell vector: ssc_h
└ @ Metacells.AnnDataFormat ~/anaconda3/envs/metacells-sharpening/share/julia/packages/Metacells/vXiuC/src/anndata_format.jl:597
┌ Debug: copy cell vector: developmental_time
└ @ Metacells.AnnDataFormat ~/anaconda3/envs/metacells-sharpening/share/julia/packages/Metacells/vXiuC/src/anndata_format.jl:597
┌ Debug: copy cell vector: Libprep.Cycles to: libprep_cycles
└ @ Metacells.AnnDataFormat ~/anaconda3/envs/metacells-sharpening/share/julia/packages/Metacells/vXiuC/src/anndata_format.jl:599
┌ Debug: copy cell vector: fsc_h
└ @ Metacells.AnnDataFormat ~/anaconda3/envs/metacells-sharpening/share/julia/packages/Metacells/vXiuC/src/anndata_format.jl:597
┌ Debug: skip cell vector: metacell
└ @ Metacells.AnnDataFormat ~/anaconda3/envs/metacells-sharpening/share/julia/packages/Metacells/vXi

name: cells
type: FilesDaf
path: /net/mraid20/ifs/wisdom/tanay_lab/data/users/obk/src/metacells-sharpening-vignette/dafs/cells
mode: w
axes:
  cell: 110746 entries
  gene: 28183 entries
vectors:
  cell:
    age_group: 110,746 x Float64 (Dense)
    age_group_emb: 110,746 x Float64 (Dense)
    alexa_fluor_488_a: 110,746 x Float64 (Dense)
    apc_a: 110,746 x Float64 (Dense)
    apc_cy7_a: 110,746 x Float64 (Dense)
    average_size_bp: 110,746 x Str (Dense)
    batch: 110,746 x Str (Dense)
    cell: 110,746 x Str (Dense)
    comment: 110,746 x Str (Dense)
    concentration_ng_per_ul: 110,746 x Str (Dense)
    coordinates: 110,746 x Str (Dense)
    delta_ct: 110,746 x Str (Dense)
    description: 110,746 x Str (Dense)
    developmental_time: 110,746 x Float64 (Dense)
    dissolved: 110,746 x Bool (Sparse 2 (<1%) [UInt32])
    embryo: 110,746 x Str (Dense)
    embryo_with_placenta_information: 110,746 x Str (Dense)
    excluded_UMIs: 110,746 x UInt32 (Dense)
    experiment: 110,746 x Str (D

## Cleaning the data


In [3]:
# How this data spells "there is no value here", which is not one way but several, sometimes several
# in the same property: `embryo` says both `NA` and nothing at all. Nothing infers these - a type
# genuinely called `NA` is possible - so each is named, and a property named here whose data happens
# to be clean is simply left alone.
#
# This has to happen before any axis is built, since building one asks of each cell whether it has a
# value: a property still saying `NA` would put `NA` on the axis, sitting among the real entries.
EMPTY_VALUES = {
    "embryo": ("NA",),
    "metacell": ("Outliers",),
    "type": ("Outliers", "Doublet"),
    "projected_type": ("(Missing)",),
    "cell": ("NA",),
    "coordinates": ("NA",),
    "source": ("NA",),
}

for property_name, empty_values in EMPTY_VALUES.items():
    dp.unify_empty_vector_values(cells, axis="cell", property=property_name, empty_values=empty_values)

# Numbers which arrived as text, because a few of their entries say `NA` and one `NA` makes a whole
# column of measurements a column of strings. Converting and unifying is one step, not two: what
# `23.5` should become is obvious, and what `NA` should become is only obvious once we are told that
# it means nothing. A number which is neither is an error rather than a silent `NaN`.
AS_NUMBERS = {
    "qc1": ("NA", np.float32),
    "qc2": ("NA", np.float32),
    "delta_ct": ("NA", np.float32),
    "concentration_ng_per_ul": ("NA", np.float32),
    # 1..32, so `0` is free to mean "none" - the convention `Daf` already uses for module indices.
    # The cells with no plate are the ones sequenced by 10x, which has no plates.
    "internal_index": ("", np.uint32),
}

for property_name, (empty_values, dtype) in AS_NUMBERS.items():
    dp.unify_empty_vector_values(
        cells, axis="cell", property=property_name, empty_values=empty_values, dtype=dtype
    )

# A sentinel which is not obviously one: the smallest 32 bit integer, which survived a cast to float
# and so is an ordinary number as far as anything reading it is concerned. Left alone, the mean of
# this property is wrong by a couple of billion rather than visibly absent.
dp.unify_empty_vector_values(
    cells, axis="cell", property="transcriptional_rank", empty_values=np.float64(-2147483648.0)
)

## Reconstructing the axes


In [4]:
# The types, and the color of each, which is what makes the graphs readable. The file decides which
# types there are and in what order they are listed - usually a meaningful order rather than an
# alphabetical one. It may name a type no cell has; a type of some cell which it does not name is an
# error, in the file or in the data. Skip this and everything still runs, uncolored.
mc.import_type_colors_csv(cells, type_colors_csv="input/type_colors.csv")

# `AnnData` has two axes, so everything else it knows is flattened onto the cells: which batch a cell
# came from, and with it every fact about that batch, repeated across its cells. Reconstructing an
# axis puts each fact where it belongs - one value per batch rather than 110,746 copies of it - and
# says so in the structure rather than in a naming convention.
#
# What is per batch, and what is merely constant within a batch by accident, is decided by looking:
# a property whose value differs between two cells of a batch is left alone. That is convenient and
# slightly dangerous, since a property which happens to be uniform is moved as readily as one which
# is uniform for a reason. These are the pipeline's own, which belong to the cells whatever their
# values happen to look like here - `is_excluded` is false for every cell of this data set, which
# says nothing about where it belongs.
KEEP_PER_CELL = {"is_excluded", "is_properly_sampled", "is_rare", "rare_gene_module", "spike_count"}

for axis in ("batch", "embryo"):
    dp.reconstruct_axis(cells, existing_axis="cell", implicit_axis=axis, skipped_properties=KEEP_PER_CELL)

# A batch was on a plate and was sequenced in a run, so those are properties of the batch now, and
# each is an axis of its own with the batch's facts divided again between them. The wet lab's record
# lands where it is read: the plate's owner and dates on the plate, the batch's concentration and QC
# on the batch, the sequencing dates on the run.
#
# The coarser axis goes first. Each plate was sequenced in one run, so a fact about a run is also
# constant within each of its plates, and reconstructing the plate first would take the run's dates
# with it - leaving the run with nothing. The reverse cannot happen: a run holds many plates, so a
# plate's own owner and dates are not constant within it.
for axis in ("sequencing_run", "plate"):
    dp.reconstruct_axis(cells, existing_axis="batch", implicit_axis=axis, skipped_properties=KEEP_PER_CELL)

# Each plate belongs to one sequencing run, but nothing has said so where a plate can be asked. It
# cannot be reconstructed: the cells sequenced by 10x have a run and no plate at all, so moving the
# run onto the plate would discard theirs. Connecting says it while leaving the batch's own run
# alone, and fails if any plate's batches disagree about which run they were in.
dp.connect_axes(cells, base_axis="batch", from_axis="plate", to_axis="sequencing_run")

print(cells.description())

┌ Debug: Metacells.AnnDataFormat.import_type_colors_csv! {
└ @ Metacells.AnnDataFormat ~/anaconda3/envs/metacells-sharpening/share/julia/packages/Metacells/vXiuC/src/anndata_format.jl:461
┌ Debug: - daf: FilesDaf cells
└ @ Metacells.AnnDataFormat ~/anaconda3/envs/metacells-sharpening/share/julia/packages/Metacells/vXiuC/src/anndata_format.jl:461
┌ Debug: - type_colors_csv: "input/type_color..." (21)
└ @ Metacells.AnnDataFormat ~/anaconda3/envs/metacells-sharpening/share/julia/packages/Metacells/vXiuC/src/anndata_format.jl:461
┌ Debug: - axis: "cell"
└ @ Metacells.AnnDataFormat ~/anaconda3/envs/metacells-sharpening/share/julia/packages/Metacells/vXiuC/src/anndata_format.jl:461
┌ Debug: - property: "type"
└ @ Metacells.AnnDataFormat ~/anaconda3/envs/metacells-sharpening/share/julia/packages/Metacells/vXiuC/src/anndata_format.jl:461
┌ Debug: - type_axis: "type"
└ @ Metacells.AnnDataFormat ~/anaconda3/envs/metacells-sharpening/share/julia/packages/Metacells/vXiuC/src/anndata_format.jl:461


name: cells
type: FilesDaf
path: /net/mraid20/ifs/wisdom/tanay_lab/data/users/obk/src/metacells-sharpening-vignette/dafs/cells
mode: w
axes:
  batch: 416 entries
  cell: 110746 entries
  embryo: 385 entries
  gene: 28183 entries
  plate: 246 entries
  sequencing_run: 195 entries
  type: 44 entries
vectors:
  batch:
    average_size_bp: 416 x Str (Dense)
    comment: 416 x Str (Dense)
    concentration_ng_per_ul: 416 x Float32 (Dense)
    delta_ct: 416 x Float32 (Dense)
    external_index: 416 x Str (Dense)
    internal_index: 416 x UInt32 (Dense)
    plate: 416 x Str (Dense)
    qc1: 416 x Float32 (Dense)
    qc2: 416 x Float32 (Dense)
    sequencing_run: 416 x Str (Dense)
  cell:
    alexa_fluor_488_a: 110,746 x Float64 (Dense)
    apc_a: 110,746 x Float64 (Dense)
    apc_cy7_a: 110,746 x Float64 (Dense)
    batch: 110,746 x Str (Dense)
    cell: 110,746 x Str (Dense)
    coordinates: 110,746 x Str (Dense)
    dissolved: 110,746 x Bool (Sparse 2 (<1%) [UInt32])
    embryo: 110,746 x S